# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Student Information

In [25]:
# Fill in your information
name = "Mehrab Jalil Abir"           # Write your full name here
email = "mehrababir1@outlook.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Mehrab Jalil Abir
Email : mehrababir1@outlook.com


---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [26]:
# Question 1
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Assignments/A2/house_price_regression_dataset.csv")
print("Shape: ",df.shape)
print(df.head(10))
df.sample(5)

Shape:  (1000, 8)
   Square_Footage  Num_Bedrooms  ...  Neighborhood_Quality   House_Price
0            1360             2  ...                     5  2.623829e+05
1            4272             3  ...                     6  9.852609e+05
2            3592             1  ...                     9  7.779774e+05
3             966             1  ...                     8  2.296989e+05
4            4926             2  ...                     8  1.041741e+06
5            3944             5  ...                     8  8.797970e+05
6            3671             1  ...                     1  8.144279e+05
7            3419             1  ...                     1  7.034131e+05
8             630             3  ...                     8  1.738750e+05
9            2185             4  ...                     5  5.041765e+05

[10 rows x 8 columns]


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
772,2469,5,1,1987,4.438031,2,9,6.005642e+05
169,3772,5,1,2005,0.570843,0,2,8.191256e+05
299,4995,3,2,2007,3.680040,0,2,1.092027e+06
308,4269,3,1,1962,4.439325,2,5,9.349421e+05
722,4824,1,1,1961,0.778486,0,1,9.240129e+05


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [27]:
#Question 2
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

x = df.drop(columns=['House_Price']);
y = df['House_Price'];

numerical_columns = ['Square_Footage','Num_Bedrooms','Num_Bathrooms','Year_Built','Lot_Size','Garage_Size'];

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# df.isnull().sum()
numerical_imputer = SimpleImputer(missing_values=np.nan,strategy='mean');
numerical_imputer.fit(x_train[numerical_columns]);
x_train[numerical_columns] = numerical_imputer.transform(x_train[numerical_columns]);

x_test[numerical_columns] = numerical_imputer.transform(x_test[numerical_columns]);

# imputing categorical values
categ_imputer = SimpleImputer(missing_values=np.nan,strategy='most_frequent');
categ_imputer.fit(x_train[['Neighborhood_Quality']])

x_train['Neighborhood_Quality'] = categ_imputer.transform(x_train[['Neighborhood_Quality']]).ravel();
x_test['Neighborhood_Quality'] = categ_imputer.transform(x_test[['Neighborhood_Quality']]).ravel();

In [28]:
print(x_train.shape)
print(x_test.shape)

(800, 7)
(200, 7)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [29]:
#Question 3
x_train_simple = x_train['Square_Footage'].to_numpy();
x_test_simple = x_test['Square_Footage'].to_numpy();
y_train_simple = y_train.to_numpy();

# Scaling Square_Footage
x_mean = np.mean(x_train_simple)
x_std = np.std(x_train_simple)

x_train_scaled = (x_train_simple - x_mean) / x_std
x_test_scaled = (x_test_simple - x_mean) / x_std

def make_prediction(x,m,c):
  n_rows = x.shape[0];
  prediction_list = np.zeros(n_rows,);

  for i in range(n_rows):
    prediction_list[i] = m * x[i] + c;
  return prediction_list;


def compute_cost(x,y,m,c):
    n_rows = x.shape[0]

    cost = 0.0

    for i in range(n_rows):
        prediction = m * x[i] + c;
        error = prediction - y[i];
        error_squared = error ** 2;
        cost = cost + error_squared;

    cost = cost/(2*n_rows);

    return cost;

def calculate_gradient(x,y,m,c):
    n_rows = x.shape[0];

    dj_dw = 0.0;
    dj_db = 0.0;

    for i in range(n_rows):
        prediction = m * x[i] + c;
        error = prediction - y[i];
        dj_dw = dj_dw + (error * x[i]);
        dj_db = dj_db + error

    dj_dw = dj_dw/n_rows;
    dj_db = dj_db/n_rows;

    return dj_dw, dj_db;


def gradient_descent(x,y,m_init,c_init,max_iter, alpha=0.00001):
    m = m_init;
    c = c_init;


    for i in range(max_iter):
        dj_dw, dj_db = calculate_gradient(x,y,m,c);

        # updating w and b
        m = m - (alpha * dj_dw);
        c = c - (alpha * dj_db);

    return m,c;

m,c = gradient_descent(x_train_scaled,y_train_simple,m_init=0.0,c_init=0.0,max_iter=10000,alpha=0.01)

y_pred = make_prediction(x_test_simple, m, c)

print("Slope (m):", m)
print("Intercept (c):", c)
print("Predictions:", y_pred)

Slope (m): 251094.52785926187
Intercept (c): 618576.0544203798
Predictions: [1.00800982e+09 5.80646935e+08 1.18277161e+09 1.23901679e+09
 9.16109225e+08 9.01043553e+08 1.16519500e+09 1.03688569e+09
 9.50006986e+08 1.06601266e+09 9.07069822e+08 7.70976588e+08
 1.96472308e+08 1.23725913e+09 6.44676040e+08 2.22837233e+08
 5.49511214e+08 4.28734746e+08 8.37265543e+08 3.46124646e+08
 8.68401264e+08 5.76629423e+08 8.33248030e+08 1.03939664e+09
 4.88997433e+08 3.65710020e+08 4.98790119e+08 4.79958030e+08
 2.30872258e+08 4.98036836e+08 2.71800666e+08 1.07103455e+09
 8.87233354e+08 3.50644348e+08 5.35701015e+08 9.06567633e+08
 5.28670368e+08 6.48693552e+08 2.45184646e+08 1.66592059e+08
 3.34574298e+08 2.52215293e+08 5.56792955e+08 3.20513005e+08
 1.25483574e+09 1.09614400e+09 9.32681463e+08 9.49253702e+08
 1.17348112e+09 8.52080120e+08 2.54224049e+08 8.44547284e+08
 9.22135493e+08 8.10147334e+08 5.79642557e+08 9.30170518e+08
 1.60565790e+08 3.19257532e+08 8.46556040e+08 2.31374447e+08
 1.159419

---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [30]:
#Question 4
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

x = df.drop(columns=['House_Price']);
y = df['House_Price'];

numerical_columns = ['Square_Footage','Num_Bedrooms','Num_Bathrooms'];

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

transformer = ColumnTransformer(
    transformers=[
        ("encoder",OneHotEncoder(sparse_output=False),['Neighborhood_Quality']),
        ("scaler",StandardScaler(),numerical_columns)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

transformer.fit(x_train)

x_train = transformer.transform(x_train);
x_test = transformer.transform(x_test);

In [31]:
df.sample(2)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
865,4951,3,1,1982,1.073686,2,3,1.017583e+06
8,630,3,3,1997,1.014286,1,8,1.738750e+05


## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [32]:
# Question 5
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score,root_mean_squared_error

x = df.drop(columns=['House_Price']);
y = df['House_Price'];

numerical_cols = ['Square_Footage','Num_Bedrooms','Num_Bathrooms','Year_Built','Lot_Size','Garage_Size'];
categ_cols = ['Neighborhood_Quality'];

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42);

numerical_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='mean')),
        ("scaler",StandardScaler())
    ]
)

categ_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("encoder",OneHotEncoder(sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical",numerical_transformer,numerical_cols),
        ("nominal",categ_transformer,categ_cols)
    ],
    remainder='passthrough'
)

sgdRegPipe = Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("model",SGDRegressor())
    ]
)

sgdRegPipe.fit(x_train,y_train);

y_prediction = sgdRegPipe.predict(x_test);

print(f"RMSE: ",root_mean_squared_error(y_test,y_prediction))
print(f"R2: ",r2_score(y_test,y_prediction))

comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy()[:10],
    "Predicted Price":y_prediction[:10]
})

print(comparison)


RMSE:  10275.821547964295
R2:  0.9983618619184579
   Actual Price  Predicted Price
0  9.010005e+05     8.698692e+05
1  4.945375e+05     4.930649e+05
2  9.494042e+05     9.434972e+05
3  1.040389e+06     1.032231e+06
4  7.940100e+05     7.761243e+05
5  7.240336e+05     7.330136e+05
6  9.984392e+05     9.919897e+05
7  9.097134e+05     8.858301e+05
8  7.926815e+05     7.961632e+05
9  9.474908e+05     9.321572e+05


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [33]:
# Question 6
# Question 5
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,root_mean_squared_error

x = df.drop(columns=['House_Price']);
y = df['House_Price'];

numerical_cols = ['Square_Footage','Num_Bedrooms','Num_Bathrooms','Year_Built','Lot_Size','Garage_Size'];
categ_cols = ['Neighborhood_Quality'];

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42);

numerical_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='mean')),
        ("scaler",StandardScaler())
    ]
)

categ_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("encoder",OneHotEncoder(sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical",numerical_transformer,numerical_cols),
        ("nominal",categ_transformer,categ_cols)
    ],
    remainder='passthrough'
)

linearRegPipe = Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("model",LinearRegression())
    ]
)

linearRegPipe.fit(x_train,y_train);

y_prediction = linearRegPipe.predict(x_test);

print(f"RMSE: ",root_mean_squared_error(y_test,y_prediction))
print(f"R2: ",r2_score(y_test,y_prediction))

comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy()[:10],
    "Predicted Price":y_prediction[:10]
})

print(comparison)


RMSE:  10250.310447408403
R2:  0.9983699856153558
   Actual Price  Predicted Price
0  9.010005e+05     8.698744e+05
1  4.945375e+05     4.930080e+05
2  9.494042e+05     9.435438e+05
3  1.040389e+06     1.032332e+06
4  7.940100e+05     7.761808e+05
5  7.240336e+05     7.330118e+05
6  9.984392e+05     9.923748e+05
7  9.097134e+05     8.857537e+05
8  7.926815e+05     7.962108e+05
9  9.474908e+05     9.320310e+05


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [ ]:
# Question 7
import numpy as np

random_100 = df.sample(100)

x = random_100[
    ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
].to_numpy(dtype=float)

y = random_100['House_Price'].to_numpy(dtype=float)

#? adding an intercept columns of 1s
x = np.c_[np.ones(x.shape[0]), x]

theta = np.linalg.inv(x.T @ x) @ (x.T @ y)

print("Intercept:", theta[0])
print("Square Footage:", theta[1])
print("Num Bedrooms:", theta[2])
print("Num Bathrooms:", theta[3])

Intercept: 13746.886065864936
Square Footage: 200.82443790007255
Num Bedrooms: 8337.898644822708
Num Bathrooms: 8890.75806610263
